[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/soumyaroy799/SUIT_solar_workshops/blob/main/workshops/USO_Vikram_2026_Oct/notebooks/lcurve.ipynb)

In [1]:
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/soumyaroy799/SUIT_solar_workshops.git"

if IN_COLAB:

    ROOT = Path("SUIT_solar_workshops")

    if not ROOT.exists():

        print("Cloning workshop repository...")

        subprocess.run(
            ["git", "clone", REPO_URL],
            check=True,
        )

    ROOT = ROOT.resolve()

else:

    ROOT = Path.cwd().resolve()

    while ROOT.name != "SUIT_solar_workshops":

        if ROOT.parent == ROOT:

            raise RuntimeError(
                "Please run inside "
                "SUIT_solar_workshops"
            )

        ROOT = ROOT.parent

import sys

sys.path.insert(
    0,
    str(ROOT),
)

In [2]:
from workshop_setup import setup_workshop

WORKSHOP_DIR, DATA_DIR = setup_workshop(ROOT)

Qt backend enabled
Workshop data already available (19 FITS files found).
Light-curve data already available (816 FITS files found).


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from glob import glob
from sunpy.map import Map
from datetime import datetime

import astropy.units as u
from astropy.coordinates import SkyCoord
from matplotlib.path import Path
from sunpy import timeseries as ts
from tqdm import *

from sunkit_image.coalignment import calculate_match_template_shift as cal_shift
from sunkit_image.coalignment import apply_shifts 

from matplotlib.lines import Line2D
from matplotlib.colors import PowerNorm
import csv
from sunpy import timeseries as ts

import warnings
warnings.filterwarnings('ignore')

#### Next up we have a couple of functions that will help us along the way. The next function exposure normalises the data (new data = old data/exposure) and changes the exposure time in header too, as well as the same RoI selection snippet we used in the previous code.


In [6]:
def exp_norm(map_sequence):
    mod_rois = []
    for smap in tqdm(map_sequence):
        esf = (smap.meta['cmd_expt'])/1e3 #exposure scaling factor. The header exp. time is saved in ms
        data = smap.data/esf   
        data[data<0] = 0
        smap.meta['cmd_expt'] = 1000 #changing the exposure to 1 sec
        mod_rois.append(Map(data,smap.meta))
    mod_rois = Map(mod_rois,sequence=True)
    return mod_rois

def start_roi_selection(sunpy_map):
    """
    Display a map and collect two mouse clicks.

    Returns
    -------
    fig, ax, roi
    """

    roi = {
        "clicks": [],
    }

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(projection=sunpy_map)

    sunpy_map.plot(axes=ax)

    ax.set_title(
        "Click two opposite corners of the ROI"
    )

    def onclick(event):

        if (
            event.xdata is None
            or event.ydata is None
        ):
            return

        roi["clicks"].append(
            (event.xdata, event.ydata)
        )

        print(
            f"Point {len(roi['clicks'])}: "
            f"({event.xdata:.1f}, "
            f"{event.ydata:.1f})"
        )

        if len(roi["clicks"]) == 2:

            (x1, y1), (x2, y2) = roi["clicks"]

            xmin = min(x1, x2)
            xmax = max(x1, x2)

            ymin = min(y1, y2)
            ymax = max(y1, y2)

            rect = patches.Rectangle(
                (xmin, ymin),
                xmax - xmin,
                ymax - ymin,
                fill=False,
                linewidth=2,
                color="red",
            )

            ax.add_patch(rect)

            fig.canvas.draw_idle()

            print(
                "\nROI selected."
                "\nRun create_submap()"
            )

    fig.canvas.mpl_connect(
        "button_press_event",
        onclick,
    )

    plt.show()

    return fig, ax, roi

def create_submap(
    sunpy_map,
    roi,
):
    """
    Create a submap from ROI clicks.
    """

    if len(roi["clicks"]) != 2:

        raise RuntimeError(
            "ROI not yet selected."
        )

    (x1, y1), (x2, y2) = roi["clicks"]

    xmin = min(x1, x2)
    xmax = max(x1, x2)

    ymin = min(y1, y2)
    ymax = max(y1, y2)

    bottom_left = u.Quantity(
        [xmin, ymin],
        u.pixel,
    )

    top_right = u.Quantity(
        [xmax, ymax],
        u.pixel,
    )

    return sunpy_map.submap(
        bottom_left=bottom_left,
        top_right=top_right,
    )